# Stock signal analysis (pipeline + dashboard inputs)
Run by `run_all.py`: at the Friday 2:30 PM CT decision (before the trade step) and after the close on the other
trading days. Scores every stock from fresh daily bars and writes the files the dashboard, forward test and trade
freshness checks read. Every number at date `t` uses only data up to `t` (no lookahead).

**Live book** = `sleeves.py` (Short-term relative strength, skip stretched buys): top 10 equal weight, Friday-only
rebalance, 99% invested (49.5% when QQQ closes at or below its 200-day average). `Strategy_Weight`,
`strategy_picks.csv`, `strategy_changes.csv` and `strategy_decisions.csv` show that book; the order sizes come from
`sleeves.plan` at the trade step. `Strategy_Score` / `Strategy_Rank` are the pipeline's low-volatility score, a
forward-test input only.

Outputs in `Reports/`: `signal_analysis.csv`, `strategy_picks.csv`, `strategy_changes.csv`, `strategy_decisions.csv`,
`strategy_midweek_check.csv` (run-log summary), `benchmark_prices.csv`, `factor_history.csv`, `short_history_reference.csv`.

### 1. Setup and prices
Daily bars from SQLite `bars_daily` (single source of truth; Alpaca Adjustment.ALL via `be.refresh_daily_bars`). Stocks with fewer than 200 trading days are set aside (not traded yet).
A decision day keeps the bar its 2:30 PM CT run traded on (`Reports/decision_bars.csv`), so later runs rebuild the same decision.

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import db  # tables live in SQLite (db.py), not CSV files

import backtest_engine as be
import sector_mapping

REPORTS_DIR = str(sector_mapping.REPORTS_DIR)
SIGNAL_CSV = os.path.join(REPORTS_DIR, "signal_analysis.csv")
FACTOR_CSV = os.path.join(REPORTS_DIR, "factor_history.csv")
PICKS_CSV = os.path.join(REPORTS_DIR, "strategy_picks.csv")
CHANGES_CSV = os.path.join(REPORTS_DIR, "strategy_changes.csv")
DECISIONS_CSV = os.path.join(REPORTS_DIR, "strategy_decisions.csv")
MIDWEEK_CSV = os.path.join(REPORTS_DIR, "strategy_midweek_check.csv")
BENCH_CSV = os.path.join(REPORTS_DIR, "benchmark_prices.csv")
RULES_VERSION = be.rules_version()   # stored with every row; older rows of the same version stay frozen
S = be.WINNER

tradable = list(sector_mapping.tradable_symbols)
display_only = [s for s in sector_mapping.stock_symbols if s not in tradable]
print(f"{len(tradable)} tradable + display-only {display_only}; strategy: {S['name']}")

start = (datetime.now() - timedelta(days=1100)).date()
# Single source of truth: refresh SQLite bars_daily from Alpaca, then score from the table (not a parallel live fetch).
_syms = list(dict.fromkeys(tradable + display_only + be.BENCHMARKS + be.SECTOR_ETFS))
try:
    _refreshed = be.refresh_daily_bars(_syms, start=be.LONG_START)
    print(f"bars_daily: refreshed {len(_refreshed)} symbols")
except Exception as e:
    print(f"WARNING bars_daily refresh failed ({e}); scoring from stored rows")
be.ensure_long_cache()   # append any brand-new tickers still missing after the refresh
bars = be.load_long_bars(symbols=_syms, start=start)
bars, dropped_partial = be.drop_partial_last_bar(bars)
bars = be.keep_decision_bars(bars)   # decision days keep their 2:30 PM bar, the price traded (Reports/decision_bars.csv)
# Short history: a stock with fewer than be.MIN_BARS (200) bars is not scored, ranked, picked or traded (and is left out of
# the relative-strength comparison). It goes to Reports/short_history_reference.csv, which the dashboard lists as
# "not traded yet"; it joins the ranking by itself on its 200th bar.
short_ref = be.short_history_reference(bars, tradable)
short_ref.reindex(columns=be.SHORT_HISTORY_COLUMNS).pipe(db.to_csv, be.SHORT_HISTORY_CSV, index=False)
if len(short_ref):
    print(f"Short history (not scored yet): {sorted(short_ref['Symbol'])}")
    _short = set(short_ref["Symbol"])
    tradable = [s for s in tradable if s not in _short]
    bars = bars[~bars["Symbol"].isin(_short)].reset_index(drop=True)
else:
    print("Short history: none")
S["tag"], S["name"] = be.winner_label(len(tradable))   # the label counts the stocks actually scored (e.g. C6-U91-...)
tech = be.build_technical(bars, symbols=tradable + display_only)
print(f"Bars through {bars['Date'].max():%Y-%m-%d} (partial dropped: {dropped_partial})")
print(f"   Missing: {sorted(set(tradable + display_only) - set(tech['Symbol']))}")


### 2. Scores
`Strategy_Score` (low volatility in uptrend, `backtest_engine.winner_score`) and `Strategy_Rank` are forward-test inputs,
not the live rules. `Regime_On` = QQQ above its 200-day average (the live market filter).
Stock list source: `sector_mapping.py` only.

In [ ]:
# --- Wide-format inputs: one column per symbol, one row per date ---
close_all = be.wide(bars, "Close")
dates = close_all.index
elig = be.bool_wide(tech, "eligible", dates, tradable)
# --- Pipeline scores (forward-test inputs, not the live rules): relative strength and Strategy_Score (be.winner_score:
# low volatility in uptrend). forward_test.py reads Strategy_Score (eligibility + several paper lines) and Regime_On.
rs_score = be.relative_strength(close_all, tradable, eligible_w=elig)   # each date ranks only the stocks eligible on it
vol63 = be.volatility(close_all[tradable])
ma200 = be.wide(tech, "ma_200").reindex(index=dates, columns=tradable)
strategy_score = be.winner_score(rs_score, close_all[tradable], ma200, vol63)
strategy_rank = be.deterministic_rank(strategy_score, elig & vol63.reindex_like(elig).gt(0), tiebreak_w=rs_score)
regime = be.regime_series(close_all, S["regime_symbol"])   # QQQ above its 200-day average (the live market filter)
earnings = be.load_earnings()
# The live book (Strategy_Weight, strategy_picks / changes / decisions) comes from sleeves.py in section 4, after
# signal_analysis.csv holds today's rows (the sleeve reads its inputs from it).
assert not S.get("midweek_swap"), "Mon-Thu checks are retired; the live line decides on Fridays only"

# --- Long-format signal panel: one row per symbol x date ---
def to_long(frame, name):
    return frame.stack(future_stack=True).rename(name).rename_axis(["Date", "Symbol"]).reset_index()

df = tech[tech["ma_200"] > 0].copy()
for frame, name in [(rs_score, "RS_Score"), (strategy_score.where(elig), "Strategy_Score"), (strategy_rank, "Strategy_Rank")]:
    df = df.merge(to_long(frame, name), on=["Date", "Symbol"], how="left")
latest_date = df["Date"].max()
df["Regime_On"] = df["Date"].map(regime.astype(int)).fillna(0).astype(int)
df["combined_signal"] = df["Strategy_Score"]
print(f"Latest {latest_date:%Y-%m-%d} | QQQ filter: {'ON' if regime.iloc[-1] else 'OFF (a Friday rebalance invests half)'}")

### 3. Fundamentals and news (shown only)
Not used by the strategy: shown on the dashboard for the latest day and saved to `factor_history.csv`.

In [ ]:
sentiment = (db.read_csv(os.path.join(REPORTS_DIR, "weighted_sentiment.csv"), usecols=["Symbol", "SentimentScore"])
             if db.exists(os.path.join(REPORTS_DIR, "weighted_sentiment.csv"))
             else pd.DataFrame(columns=["Symbol", "SentimentScore"]))   # fresh DB: no news yet -> neutral 0 below
fundamentals = (db.read_csv(os.path.join(REPORTS_DIR, "balance_sheet_weights.csv"), usecols=["Symbol", "Fundamental_Weight"])
                if db.exists(os.path.join(REPORTS_DIR, "balance_sheet_weights.csv"))
                else pd.DataFrame(columns=["Symbol", "Fundamental_Weight"]))   # fresh DB: stays empty
# Refuse to merge if either file has duplicate symbols (would corrupt the panel).
for _frame, _name in [(sentiment, "weighted_sentiment.csv"), (fundamentals, "balance_sheet_weights.csv")]:
    _dup = _frame.loc[_frame["Symbol"].duplicated(), "Symbol"].unique()
    if len(_dup):
        raise ValueError(f"{_name} has duplicate Symbol rows: {sorted(_dup)}")
is_latest = df["Date"].eq(latest_date)
df = df.merge(sentiment, on="Symbol", how="left").merge(fundamentals, on="Symbol", how="left")
# No news = neutral (0); missing fundamentals stay empty. Only the latest date keeps values.
df.loc[is_latest, "SentimentScore"] = df.loc[is_latest, "SentimentScore"].fillna(0.0)
df.loc[~is_latest, ["SentimentScore", "Fundamental_Weight"]] = np.nan
# Append today's snapshot to the factor history (for future backtests).
snap = df.loc[is_latest, ["Date", "Symbol", "SentimentScore", "Fundamental_Weight", "RS_Score", "Technical_Score",
                          "Strategy_Score"]].rename(columns={"Date": "bar_date"})
snap.insert(0, "as_of", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))
history = db.read_csv(FACTOR_CSV, parse_dates=["bar_date"]) if db.exists(FACTOR_CSV) else pd.DataFrame()
history = pd.concat([history, snap], ignore_index=True).drop_duplicates(["bar_date", "Symbol"], keep="last")
# Only the stocks still in the list and only the columns above (older files had sector-score columns nothing reads).
history = history[history["Symbol"].isin(set(df["Symbol"]))].reindex(columns=snap.columns)
history.round(4).pipe(db.to_csv, FACTOR_CSV, index=False)
print(f"factor_history.csv: {len(history)} rows")

### 4. Signals and saved files
BUY = in the live book · SELL = not held and score below 0 · HOLD = otherwise · BENCHMARK = a comparison symbol in the stock list (QQQ), never traded. Then every output file is written.

In [ ]:
import sleeves
OUTPUT_COLUMNS = [   # signal_analysis.csv schema (paper_trade.py, forward_test.py and the app depend on these columns)
    "Date", "Symbol", "Close", "ma_10", "ma_30", "ma_50", "ma_100", "ma_200",
    "RSI", "macd", "MACD Signal", "Technical_Score", "SentimentScore", "Fundamental_Weight",
    "combined_signal", "final_trade",
    "is_earnings_date", "RS_Score", "Strategy_Score", "Strategy_Rank", "Strategy_Weight",
    "Provisional_Weight", "Regime_On", "Rebalance_Day", "rules_version", "Midweek_Check",
]
LIVE_RULES = RULES_VERSION + "-live-rs"   # Strategy_Weight / final_trade follow the live sleeve (Nov 8, 2026)
today = pd.Timestamp(latest_date).normalize()
window_end = be.next_sessions(today - pd.Timedelta(days=1), 3)[-1]
near_earnings = earnings.loc[earnings["Earnings Date"].between(today, window_end), "Symbol"]
prev = db.read_csv(SIGNAL_CSV) if db.exists(SIGNAL_CSV) else pd.DataFrame()   # before the first write below

def signal_table(live_w=None, weekly=None):
    """signal_analysis.csv rows; live_w (Date x Symbol live weights) / weekly (the live Friday calendar) None = blank."""
    t = df.sort_values(["Symbol", "Date"]).reset_index(drop=True)
    w = to_long(live_w, "Strategy_Weight") if live_w is not None else None
    t = t.merge(w, on=["Date", "Symbol"], how="left") if w is not None else t.assign(Strategy_Weight=np.nan)
    t["Provisional_Weight"] = t["Strategy_Weight"].where(t["Date"] == latest_date)
    t["Rebalance_Day"] = t["Date"].map(weekly.astype(int)).fillna(0).astype(int) if weekly is not None else 0
    t["Midweek_Check"] = 0                                          # no Mon-Thu checks (retired Nov 4, 2026)
    # BUY = in the live book, SELL = score below 0, HOLD = otherwise, BENCHMARK = QQQ (display only, never traded).
    rule = pd.Series("HOLD", index=t.index)
    rule = rule.mask(t["Strategy_Score"] < S["min_score"], "SELL").mask(t["Strategy_Weight"].fillna(0) > 0, "BUY")
    rule = rule.mask(t["Symbol"].isin(display_only), "BENCHMARK")
    t["final_trade"] = rule
    if "rules_version" in prev.columns:            # older rows of the same rules keep their label (stable history)
        p = prev.assign(Date=pd.to_datetime(prev["Date"]).dt.normalize())
        p = p[(p["rules_version"] == LIVE_RULES) & (p["Date"] < p["Date"].max())]
        p = p.drop_duplicates(["Symbol", "Date"], keep="last")[["Symbol", "Date", "final_trade"]]
        t = t.merge(p.rename(columns={"final_trade": "prev_trade"}), on=["Symbol", "Date"], how="left")
        t["final_trade"] = t.pop("prev_trade").fillna(t["final_trade"])
    out = t.rename(columns={"rsi": "RSI", "macd_signal": "MACD Signal"})
    out["rules_version"] = LIVE_RULES
    out["is_earnings_date"] = pd.MultiIndex.from_frame(out[["Symbol", "Date"]]).isin(
        pd.MultiIndex.from_frame(earnings[["Symbol", "Earnings Date"]])).astype(int)
    out.loc[(out["Date"] == latest_date) & out["Symbol"].isin(near_earnings), "final_trade"] = "EARNING"   # display only
    return out[OUTPUT_COLUMNS].round(4).sort_values(["Symbol", "Date"], ascending=[True, False])

# 1) today's rows first: the live sleeve (sleeves.inputs -> forward_test.inputs) reads them from signal_analysis.csv
signal_table().pipe(db.to_csv, SIGNAL_CSV, index=False)
# 2) the live book: RS short-term (sleeves.py), the forward-test weights the account trades (QQQ filter included)
H = sleeves.HALVES[0]
sl_inp = sleeves.inputs()
live_w = sleeves.targets(H, sl_inp).reindex(index=dates, columns=tradable).fillna(0.0)
live_weekly = sl_inp["weekly"].reindex(dates, fill_value=False).astype(bool)   # the week's last session
out = signal_table(live_w.where(live_w > 0), live_weekly)
out.pipe(db.to_csv, SIGNAL_CSV, index=False)
sleeves.write_strategy_picks(latest_date, PICKS_CSV, inp=sl_inp, led=sleeves.load_ledger())
picks = db.read_csv(PICKS_CSV)

# --- strategy_decisions.csv / strategy_changes.csv: adds / holds / drops of the live book at each Friday rebalance ---
score = sl_inp["scores"][sleeves._cfg(H).get("score", "live")]
reb_days = [d for d in dates[live_weekly.to_numpy()] if d in live_w.index]
rows = []
for d in reb_days:
    i = dates.get_loc(d)
    old = live_w.iloc[i - 1] if i else live_w.iloc[i] * 0
    new = live_w.loc[d]
    rk = sleeves.ranking(H, d, sl_inp)
    pos = {s: n + 1 for n, s in enumerate(rk.index)}
    for s in sorted(set(old[old > 0].index) | set(new[new > 0].index)):
        status = "hold" if old[s] > 0 and new[s] > 0 else "add" if new[s] > 0 else "drop"
        r = pos.get(s)
        why = ({"add": f"picked at rank {r}", "hold": f"kept at rank {r}"}.get(status) if r else None) or (
            f"rank {r}: not in the new top 10" if r else "score below 0 / not eligible")
        rows.append({"Date": d, "Symbol": s, "Status": status, "Reason": why, "Rank": r,
                     "Score": score.at[d, s] if s in score.columns and d in score.index else None,
                     "Sector": sector_mapping.symbol_sector.get(s, ""), "Old_Weight": float(old[s]),
                     "New_Weight": float(new[s]), "Regime_On": int(regime.get(d, 0)), "Check": "weekly"})
dec = pd.DataFrame(rows, columns=["Date", "Symbol", "Status", "Reason", "Rank", "Score", "Sector", "Old_Weight",
                                  "New_Weight", "Regime_On", "Check"])
dec.drop(columns=["Sector", "Check"]).round(4).pipe(db.to_csv, DECISIONS_CSV, index=False)
last_rebalance = reb_days[-1] if reb_days else latest_date
changes = dec[dec["Date"] == last_rebalance].drop(columns="Regime_On").assign(View="last rebalance")
if latest_date != last_rebalance:   # the trade step's freshness check needs today's date (unknown status = HOLD)
    changes = pd.concat([changes, pd.DataFrame([{"Date": latest_date, "Symbol": "", "Status": "no_action",
                                                 "Reason": "no decision today", "New_Weight": 0.0, "Sector": "",
                                                 "View": "last rebalance", "Check": "none"}])], ignore_index=True)
changes.round(4).pipe(db.to_csv, CHANGES_CSV, index=False)

# --- strategy_midweek_check.csv: the run-log summary (last Friday rebalance + the next decision) ---
def _day(d):
    t = pd.Timestamp(d)
    return f"{t:%a %b} {t.day}"
def _orders(d, fill):
    return f"orders go out {_day(d)} at {be.slot_text(d)}, any rest {_day(fill)} at 9 AM CT"
nxt_d, nxt_kind, nxt_fill = be.next_decision(latest_date)
f0 = be.next_sessions(last_rebalance, 1)[0]
rb = changes[changes["Date"] == last_rebalance]
inv = float(live_w.loc[last_rebalance].sum()) if last_rebalance in live_w.index else 0.0
mwc = pd.DataFrame([{
    "As_Of": latest_date, "Event": "full rebalance", "Event_Date": last_rebalance, "Applies_To_Open": f0, "Action": "REBALANCE",
    "Sell": ", ".join(sorted(rb.loc[rb["Status"] == "drop", "Symbol"])),
    "Buy": ", ".join(sorted(rb.loc[rb["Status"] == "add", "Symbol"])),
    "Message": (f"{sleeves.STRATEGY[H]}: full rebalance at the {_day(last_rebalance)} close to the top 10 "
                f"({(rb['Status'] == 'add').sum()} buy, {(rb['Status'] == 'drop').sum()} sell, {(rb['Status'] == 'hold').sum()} keep; "
                f"{inv:.1%} invested{', QQQ filter off' if not regime.get(last_rebalance, True) else ''}); "
                f"{_orders(last_rebalance, f0)}."),
    "Is_Latest": 1,
    "Next_Message": f"Next decision: {nxt_kind} at {_day(nxt_d)} {be.slot_text(nxt_d)}; {_orders(nxt_d, nxt_fill)}.",
    "Rules": S.get("tag", "")}])
mwc.round(4).pipe(db.to_csv, MIDWEEK_CSV, index=False)

# --- benchmark_prices.csv: SPY/QQQ/sector ETF closes for the app's RS lines ---
# Append-only (db.append_benchmark_prices): unique on Date; never drops older history. New dates are
# inserted; only the newest session is replaced so a same-day refresh can fill blanks.
bench_cols = [c for c in be.BENCHMARKS + be.SECTOR_ETFS if c in close_all.columns]
new_bench = close_all.loc[close_all.index >= df["Date"].min(), bench_cols].round(4).rename_axis("Date").reset_index()
db.append_benchmark_prices(new_bench, BENCH_CSV)
print(f"Saved {len(out)} rows, {out['Symbol'].nunique()} symbols, latest {latest_date:%Y-%m-%d}; "
      f"live book {sleeves.STRATEGY[H]}: {len(picks)} picks, last rebalance {last_rebalance:%Y-%m-%d}")
picks